# Task 2: Molecular Descriptors (2D)
**Tool: RDKit** (open-source cheminformatics library)

This notebook computes:
- **Lipinski Rule of Five** descriptors: MW, LogP, HBD, HBA
- **Extended 2D descriptors**: TPSA, Rotatable Bonds, Aromatic Rings, Fsp3, QED
- Lipinski compliance analysis and visualizations

> If RDKit is not installed, the notebook loads pre-computed descriptors from CSV.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

try:
    from rdkit import Chem
    from rdkit.Chem import Descriptors, rdMolDescriptors, QED
    RDKIT = True
    print('✅ RDKit loaded')
except ImportError:
    RDKIT = False
    print('⚠️  RDKit not found — will load pre-computed CSV')

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120

## 2.1 Load Dataset

In [ ]:
df = pd.read_csv('../data/bioactivity_dataset.csv')
df['standard_value'] = df['standard_value'].replace(0, 0.001)
df['pIC50'] = (9 - np.log10(df['standard_value'])).round(4)
print(f'Loaded {len(df)} compounds')
df.head()

## 2.2 Compute 2D Descriptors with RDKit

### Descriptor definitions
| Descriptor | RDKit Function | Lipinski Limit |
|-----------|---------------|----------------|
| MW | `Descriptors.MolWt` | ≤ 500 Da |
| LogP | `Descriptors.MolLogP` | ≤ 5 |
| HBD | `rdMolDescriptors.CalcNumHBD` | ≤ 5 |
| HBA | `rdMolDescriptors.CalcNumHBA` | ≤ 10 |
| TPSA | `rdMolDescriptors.CalcTPSA` | — |
| RotBonds | `rdMolDescriptors.CalcNumRotatableBonds` | — |
| ArRings | `rdMolDescriptors.CalcNumAromaticRings` | — |
| Fsp3 | `rdMolDescriptors.CalcFractionCSP3` | — |
| QED | `QED.qed` | — |

In [ ]:
if RDKIT:
    def compute_descriptors(smiles):
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return {k: np.nan for k in ['MW','LogP','HBD','HBA','TPSA','RotBonds','ArRings','Fsp3','QED']}
        return {
            'MW':       round(Descriptors.MolWt(mol), 3),
            'LogP':     round(Descriptors.MolLogP(mol), 3),
            'HBD':      rdMolDescriptors.CalcNumHBD(mol),
            'HBA':      rdMolDescriptors.CalcNumHBA(mol),
            'TPSA':     round(rdMolDescriptors.CalcTPSA(mol), 3),
            'RotBonds': rdMolDescriptors.CalcNumRotatableBonds(mol),
            'ArRings':  rdMolDescriptors.CalcNumAromaticRings(mol),
            'Fsp3':     round(rdMolDescriptors.CalcFractionCSP3(mol), 4),
            'QED':      round(QED.qed(mol), 4),
        }
    from tqdm import tqdm
    desc_list = [compute_descriptors(s) for s in tqdm(df['smiles'], desc='Computing descriptors')]
    desc_df = pd.DataFrame(desc_list)
    for col in desc_df.columns:
        df[col] = desc_df[col].values
    print('✅ RDKit descriptors computed')
else:
    df = pd.read_csv('../data/descriptors_2D.csv')
    print('✅ Pre-computed descriptors loaded')

df.head()

## 2.3 Lipinski Rule of Five Analysis

In [ ]:
df['Ro5_violations'] = ((df['MW'] > 500).astype(int) +
                        (df['LogP'] > 5).astype(int) +
                        (df['HBD'] > 5).astype(int) +
                        (df['HBA'] > 10).astype(int))
df['Lipinski_pass'] = df['Ro5_violations'].apply(lambda x: 'Pass' if x <= 1 else 'Fail')

print('--- Descriptor Summary ---')
print(df[['MW','LogP','HBD','HBA','TPSA','RotBonds','QED']].describe().round(2))
print('\n--- Lipinski Compliance ---')
print(df['Lipinski_pass'].value_counts())
df.to_csv('../data/descriptors_2D.csv', index=False)
print('\n✅ Descriptor CSV saved.')

## 2.4 Descriptor Distribution Plots

In [ ]:
colors = {'active': '#2ecc71', 'inactive': '#e74c3c', 'intermediate': '#f39c12'}
fig, axes = plt.subplots(2, 3, figsize=(17, 11))
fig.suptitle('2D Molecular Descriptor Distributions by Class', fontsize=15, fontweight='bold')

desc_info = [
    ('MW',      'Molecular Weight (Da)',  500,  (0, 800)),
    ('LogP',    'LogP',                   5,    (-5, 12)),
    ('HBD',     'H-Bond Donors',          5,    (0, 12)),
    ('HBA',     'H-Bond Acceptors',       10,   (0, 16)),
    ('TPSA',    'TPSA (Å²)',             None, (0, 210)),
    ('QED',     'Drug-likeness (QED)',   None, (0, 1.05)),
]

for ax, (col, label, threshold, xlim) in zip(axes.flatten(), desc_info):
    for cls, grp in df.groupby('bioactivity_class'):
        ax.hist(grp[col], bins=40, alpha=0.50, color=colors[cls],
                label=f'{cls} (n={len(grp)})', edgecolor='none')
    if threshold:
        ax.axvline(threshold, color='black', linestyle='--', lw=1.5,
                   label=f'Ro5 limit ({threshold})')
    ax.set_xlabel(label, fontsize=10)
    ax.set_ylabel('Count', fontsize=10)
    ax.set_title(label, fontsize=11, fontweight='bold')
    ax.set_xlim(xlim)
    ax.legend(fontsize=8)
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../figures/task2_descriptor_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved ✅')

In [ ]:
# Correlation heatmap
fig, ax = plt.subplots(figsize=(10, 8))
cols = ['pIC50','MW','LogP','HBD','HBA','TPSA','RotBonds','ArRings','Fsp3','QED']
corr = df[cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, ax=ax, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.5, annot_kws={'size': 9},
            cbar_kws={'label': 'Pearson r'})
ax.set_title('Descriptor Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../figures/task2_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Task 2 Complete!')